# **Modelos Alternativos frente a XGBoost**


## **Búsqueda de modelo original**


Con el fin de explorar si existe algún enfoque capaz de superar al XGBoost clásico en la tarea multiclase —o bien mantener un F1-score macro comparable con un menor costo computacional— se implementaron tres algoritmos avanzados que hasta ahora no se habían aplicado al contexto de violencia de género en Colombia. En primer lugar, se utilizó **HistGradientBoostingClassifier**, que optimiza los cortes de las variables continuas mediante histogramas y suele ofrecer un entrenamiento más rápido que otros boosting convencionales. A continuación, se evaluó **EasyEnsembleClassifier**, un método de bagging que crea múltiples subconjuntos balanceados por undersampling y los combina en un ensamblado robusto, con la expectativa de mejorar la detección de la clase minoritaria. Por último, se testeó **BalancedBaggingClassifier**, cuyo principio consiste en aplicar undersampling interno en cada árbol del ensemble para corregir el desbalance de forma dinámica durante el entrenamiento.

Cada uno de estos modelos se integró en un pipeline con escalado de variables y validación cruzada estratificada de 5 folds, aplicando SMOTE en cada partición de entrenamiento. El criterio de comparación principal fue el F1-score macro, acompañado de la AUC macro y el tiempo de cómputo necesario. De esta manera, buscamos identificar no solo qué técnica logra el puntaje más alto en la clasificación de las tres categorías de desenlace, sino también cuál reduce la latencia de entrenamiento sin sacrificar la capacidad predictiva.

A continuación se detallan, de forma individual, los pipelines de cada modelo, los parámetros seleccionados y los resultados obtenidos en las métricas clave.


#### **Importación de librerías necesarias**

In [ ]:
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from xgboost import XGBClassifier
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import display, HTML
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from xgboost import XGBClassifier
from lime.lime_tabular import LimeTabularExplainer
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import display, HTML
import pandas as pd
import numpy as np
import time


### **HistGradientBoostingClassifier**

El **HistGradientBoostingClassifier** es una implementación de gradient boosting basada en histogramas, diseñada para optimizar tanto la velocidad de entrenamiento como la calidad de las predicciones en grandes volúmenes de datos. En lugar de buscar cortes exactos en cada observación, este método agrupa los valores continuos en bins (histogramas) y calcula las divisiones óptimas de forma eficiente, reduciendo drásticamente el coste computacional. Además, soporta `early_stopping` para detener el entrenamiento cuando la mejora en la función de pérdida se estanca, y permite aplicar `class_weight='balanced'` para corregir el sesgo hacia la clase mayoritaria en escenarios desbalanceados.

En nuestro pipeline, primero escalamos las variables con un `StandardScaler` (sin copia de datos), luego entrenamos el HistGradientBoostingClassifier sobre el conjunto balanceado con SMOTE, y finalmente optimizamos dos hiperparámetros clave —`max_iter` (número de iteraciones) y `max_leaf_nodes` (número máximo de hojas)— mediante validación cruzada estratificada de 5 folds. A continuación se muestra el código de configuración y los resultados correspondientes.  


In [ ]:
import time
import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
from imblearn.over_sampling import SMOTE
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import HTML

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# SMOTE solo sobre el train
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

memory = Memory("./cache_dir", verbose=0)

# Pipeline: scaler + HGB
hgb_pipeline = Pipeline([
    ('scaler', StandardScaler(copy=False)),
    ('hgb', HistGradientBoostingClassifier(
        early_stopping=True,
        class_weight='balanced'
    ))
], memory=memory)

# Grid de hiperparámetros
param_grid = {
    'hgb__max_iter': [100, 200],
    'hgb__max_leaf_nodes': [31, 63]
}

grid_search = GridSearchCV(
    estimator=hgb_pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1,
    error_score='raise'
)

# Entrenamiento
start_cpu = time.process_time()
grid_search.fit(X_train_smote, y_train_smote)
cpu_elapsed = time.process_time() - start_cpu

best_model  = grid_search.best_estimator_
best_params = grid_search.best_params_
print("Mejores parámetros:", best_params)
print(f"CPU time (s): {cpu_elapsed:.2f}")

# Predicciones
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

# Métricas y reportes
report = classification_report(y_test, y_pred, output_dict=True)
df_rep = pd.DataFrame(report).transpose()
f1_macro = report['macro avg']['f1-score']
accuracy = report['accuracy']

# AUC-macro multiclase
y_test_bin    = label_binarize(y_test, classes=np.unique(y))
roc_auc_macro = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

# Matriz de confusión (Plotly)
cm = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))
z_text = [[str(val) for val in row] for row in cm]
fig_cm = ff.create_annotated_heatmap(
    z=cm, x=[f"Pred_{l}" for l in labels], y=[f"True_{l}" for l in labels],
    annotation_text=z_text, colorscale='Purples'
)
fig_cm.update_layout(
    title="Matriz de Confusión - HistGradientBoosting",
    xaxis=dict(title="Predicción"), yaxis=dict(title="Valor Real"),
    font=dict(family="Inter")
)
fig_cm.show()

# Curva ROC micro-average
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
roc_micro = auc(fpr, tpr)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    name=f"ROC micro-average (AUC={roc_micro:.4f})",
    line=dict(color='darkmagenta', width=3)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    name='Azar', line=dict(color='gray', dash='dash')
))
fig_roc.update_layout(
    title="Curva ROC micro-average - HistGradientBoosting",
    xaxis_title="False Positive Rate", yaxis_title="True Positive Rate",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family="Inter", size=13, color="black"),
    width=700, height=500
)
fig_roc.show()

# Resumen
summary = {
    'Precision (Macro)': [report['macro avg']['precision']],
    'Recall (Macro)'   : [report['macro avg']['recall']],
    'F1-score (Macro)' : [f1_macro],
    'Accuracy'         : [accuracy],
    'AUC (Macro)'      : [roc_auc_macro],
    'CPU Time (s)'     : [cpu_elapsed]
}
summary_df = pd.DataFrame(summary)
display(HTML("<h2>Resumen de Métricas - HistGradientBoosting</h2>"))
display(summary_df.style.format("{:.4f}"))


Mejores parámetros: {'hgb__max_iter': 200, 'hgb__max_leaf_nodes': 63}
CPU time (s): 296.07


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC (Macro),CPU Time (s)
0,0.4808,0.4316,0.4474,0.8425,0.7505,296.0658


Tras ajustar `max_iter` a 200 y `max_leaf_nodes` a 63, el HistGradientBoostingClassifier alcanzó un tiempo de cómputo de **296.07 s**, lo que confirma el coste relativamente alto de este método a pesar de su estructura basada en histogramas y del early stopping aplicado.  

La **matriz de confusión** revela que el modelo discrimina muy bien la clase 0 (víctimas vivas y no hospitalizadas), con 79 659 verdaderos negativos y solo 4 632 falsos positivos. Para la clase intermedia (hospitalización), consigue 6 061 verdaderos positivos frente a 11 160 falsos negativos y 43 casos asignados erróneamente a la clase 2. Sin embargo, la clase 2 (fallecimiento) no se identifica en absoluto: de los 66 ejemplos reales, solo 47 se confunden como clase 0 y 19 como clase 1, y ninguno es detectado como verdadero positivo. Este desequilibrio en el recall de las categorías de más alto riesgo subraya la dificultad de separar adecuadamente los desenlaces más graves.  

La **curva ROC micro-average** muestra un AUC de **0.9583**, reflejo del excelente desempeño global impulsado por la clase mayoritaria, aunque no garantiza un buen comportamiento para las clases minoritarias por separado.  

En términos de métricas agregadas, el modelo alcanza un **F1-score macro de 0.4474**, un **recall macro de 0.4316** y una **precisión macro de 0.4808**, con una **exactitud global del 84.25 \%** y un **AUC macro de 0.7505**. Estos valores confirman que, si bien el HistGradientBoostingClassifier mantiene un poder predictivo similar al de XGBoost clásico, su elevado tiempo de entrenamiento y la incapacidad para reconocer casos de fallecimiento justifican explorar enfoques alternativos, o bien la transformación a una clasificación binaria para concentrar el aprendizaje en la detección de riesgo.  


### **EasyEnsembleClassifier**

El **EasyEnsembleClassifier** es una técnica de bagging especialmente diseñada para problemas con desequilibrio marcado. En lugar de aplicar SMOTE, este método genera múltiples subconjuntos de entrenamiento balanceados mediante undersampling de la clase mayoritaria y entrena un clasificador en cada uno de ellos. Posteriormente, combina las predicciones de todos los estimadores por votación mayoritaria, lo que aporta robustez frente a la escasez de ejemplos de la clase minoritaria. En nuestro pipeline, escalamos previamente las características con `StandardScaler` y ajustamos `n_estimators` (número de subconjuntos) mediante `GridSearchCV` con validación cruzada de 5 folds y `scoring='f1_macro'`. A continuación se muestra el código empleado y, más adelante, los resultados obtenidos en métricas clave, matriz de confusión, curva ROC y explicación local con LIME para interpretar el comportamiento del modelo.  


In [ ]:
from joblib import Memory
memory = Memory("./cache_dir", verbose=0)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
import time
import numpy as np
import pandas as pd
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import HTML

eec_pipeline = Pipeline([
    ('scaler', StandardScaler(copy=False)),
    ('eec', EasyEnsembleClassifier(random_state=42))
], memory=memory)

param_grid = {
    'eec__n_estimators': [10, 20]
}

grid_search = GridSearchCV(
    estimator=eec_pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

start_time = time.process_time()
grid_search.fit(X_train_smote, y_train_smote)
training_time = time.process_time() - start_time

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
print("Mejores parámetros encontrados:", best_params)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

report = classification_report(y_test, y_pred, output_dict=True)
accuracy_global = report["accuracy"]

unique_classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=unique_classes)
roc_auc_macro = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

cm = confusion_matrix(y_test, y_pred)
labels = sorted(unique_classes)
z_text = [[str(val) for val in row] for row in cm]
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labels],
    y=[f"True_{l}" for l in labels],
    annotation_text=z_text,
    colorscale='Purples'
)
fig_cm.update_layout(
    title_text="Matriz de Confusión - EasyEnsemble",
    xaxis=dict(title="Predicción"),
    yaxis=dict(title="Valor Real"),
    font=dict(family="Inter")
)
fig_cm.show()

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
roc_auc_micro = auc(fpr, tpr)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f"ROC micro-average (AUC={roc_auc_micro:.4f})",
    line=dict(color='darkmagenta', width=3)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    name='Azar',
    line=dict(color='gray', dash='dash')
))
fig_roc.update_layout(
    title="Curva ROC micro-average - EasyEnsemble",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Inter", size=13, color="black"),
    width=700, height=500
)
fig_roc.show()

precision_macro = report['macro avg']['precision']
recall_macro = report['macro avg']['recall']
f1_macro = report['macro avg']['f1-score']

summary_data = {
    'Precision (Macro)': [precision_macro],
    'Recall (Macro)':    [recall_macro],
    'F1-score (Macro)':  [f1_macro],
    'Accuracy':          [accuracy_global],
    'AUC (Macro)':       [roc_auc_macro],
    'Training Time (s)': [training_time]
}
summary_df = pd.DataFrame(summary_data)
display(HTML("<h2>Resumen de Métricas - EasyEnsemble</h2>"))
display(summary_df.style.format("{:.4f}"))


Mejores parámetros encontrados: {'eec__n_estimators': 10}


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC (Macro),Training Time (s)
0,0.3857,0.5036,0.3688,0.6345,0.6849,597.0866


Tras optimizar el número de estimadores a 10, el EasyEnsembleClassifier requirió **597.09 s** de CPU, convirtiéndose en el método más costoso de esta ronda.  

La **matriz de confusión** revela un comportamiento claramente sesgado hacia la clase mayoritaria: de los 84 409 casos con desenlace 0 (“viva y no hospitalizada”), el modelo detecta correctamente 56 884 ejemplos (67.4 %) y confunde 20 548 como hospitalización y 6 977 como fallecimiento. Para los casos intermedios de hospitalización (clase 1), se obtienen 7 509 verdaderos positivos (recall ≈ 44.4 %) frente a 7 647 falsos negativos y 2 108 erróneos hacia la clase 2. En la categoría de fallecimiento (clase 2), el desempeño es aún más deficiente: solo 26 de los 66 ejemplos reales son correctamente clasificados (recall ≈ 39.4 %), mientras que 22 quedan en clase 0 y 18 en clase 1.  

La **curva ROC micro-average** (AUC ≈ 0.8111) indica una capacidad de discriminación moderada cuando se combinan todas las clases, pero este valor no se traduce en un equilibrio real entre precisión y recall para las categorías minoritarias.  

En el **resumen de métricas** destaca un F1-score (macro) de **0.3688**, una precisión macro de **0.3857** y un recall macro de **0.5036**, con una exactitud global de **0.6345** y un AUC macro de **0.6849**. Estos resultados quedan muy por debajo del F1-score macro de XGBoost (0.6792) y del recall equilibrado de Random Forest (0.6663), a la vez que su elevado tiempo de cómputo limita su viabilidad práctica.  

En consecuencia, aunque EasyEnsembleClassifier aporta cierta robustez frente al desbalance, su bajo F1 macro—sobre todo en la detección de víctimas fallecidas—y su alta demanda de recursos de cómputo lo sitúan por debajo de las opciones de referencia.  


### **BalancedBaggingClassifier**

El **BalancedBaggingClassifier** es un método de ensamble basado en Bagging que incorpora de forma nativa el muestreo aleatorio de la clase mayoritaria para equilibrar cada subconjunto de entrenamiento. A diferencia de técnicas que requieren SMOTE externo, este algoritmo genera internamente múltiples muestras balanceadas mediante *undersampling* antes de entrenar cada árbol de decisión, reduciendo el sesgo hacia la clase con más ejemplos.  

En nuestra implementación, construimos un pipeline que primero escala las variables con `StandardScaler` y luego aplica `BalancedBaggingClassifier` con un estimador base de `DecisionTreeClassifier(max_depth=6)`. Se configuran 50 estimadores y `sampling_strategy='auto'`, de modo que cada árbol se entrena sobre un conjunto distinto y equilibrado. El modelo aprovecha el paralelismo (`n_jobs=-1`) para acelerar el entrenamiento, mientras que la semilla (`random_state=42`) garantiza la reproducibilidad.  

Para evaluar su desempeño, se dividió el conjunto original en 80 % entrenamiento y 20 % prueba usando un split estratificado, se midió el tiempo de CPU con `time.process_time()` y, tras el ajuste, se calcularon las métricas clave: informe de clasificación, matriz de confusión y curva ROC *micro-average*. En los apartados siguientes se mostrará el código completo y se interpretarán las salidas obtenidas.  


In [ ]:
from joblib import Memory
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, f1_score, roc_auc_score
)
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import HTML

X_np = X.values if hasattr(X, "values") else X
y_np = y.values if hasattr(y, "values") else y

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, stratify=y_np, random_state=42
)

# Pipeline BalancedBaggingClassifier con muestreo automático
memory = Memory("./cache_dir", verbose=0)
pipe_bbc = Pipeline([
    ('scaler', StandardScaler(copy=False)),
    ('bbc', BalancedBaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6),
        sampling_strategy='auto',
        n_estimators=50,
        n_jobs=-1,
        random_state=42
    ))
], memory=memory)

# Entrenamiento y CPU time
start_cpu = time.process_time()
pipe_bbc.fit(X_train, y_train)
cpu_time = time.process_time() - start_cpu
print(f"CPU Time: {cpu_time:.1f} s")

# Predicción y probabilidades
y_pred = pipe_bbc.predict(X_test)
y_proba = pipe_bbc.predict_proba(X_test)

# Métricas
report       = classification_report(y_test, y_pred, output_dict=True)
accuracy     = report['accuracy']
f1_macro     = report['macro avg']['f1-score']
y_test_bin   = label_binarize(y_test, classes=np.unique(y_test))
roc_auc_macro = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

# Matriz de Confusión
cm     = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))
z_text = [[str(v) for v in row] for row in cm]
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labels],
    y=[f"True_{l}" for l in labels],
    annotation_text=z_text,
    colorscale='Purples'
)
fig_cm.update_layout(
    title_text="Matriz de Confusión - BalancedBaggingClassifier",
    xaxis=dict(title="Predicción"),
    yaxis=dict(title="Valor Real"),
    font=dict(family="Inter")
)
fig_cm.show()

# Curva ROC micro-average
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
roc_micro = auc(fpr, tpr)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    name=f"ROC micro-average (AUC={roc_micro:.4f})",
    line=dict(width=3)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    name='Azar', line=dict(dash='dash')
))
fig_roc.update_layout(
    title="Curva ROC micro-average - BalancedBaggingClassifier",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family="Inter", size=13),
    width=700, height=500
)
fig_roc.show()

# Resumen de métricas
summary = {
    'Accuracy':         [accuracy],
    'F1-score (Macro)': [f1_macro],
    'AUC (Macro)':      [roc_auc_macro],
    'CPU Time (s)':     [cpu_time]
}
summary_df = pd.DataFrame(summary)
display(HTML("<h2>Resumen - BalancedBaggingClassifier</h2>"))
display(summary_df.style.format("{:.4f}"))


CPU Time: 0.4 s


,Accuracy,F1-score (Macro),AUC (Macro),CPU Time (s)
0,0.6039,0.3679,0.4753,0.3632


Tras entrenar el BalancedBaggingClassifier con 50 estimadores y un árbol base de profundidad 6, el modelo completó su ajuste en apenas **0.36 s**, siendo el método más veloz de esta ronda.  

La matriz de confusión revela un comportamiento sesgado hacia la clase mayoritaria: de los **84 409** casos de la categoría 0 (“viva y no hospitalizada”), el clasificador identifica correctamente **51 899** ejemplos (61.6 %) y confunde **25 451** con hospitalización y **7 059** con fallecimiento. Para la clase 1 (“viva y hospitalizada”), reconoce **9 503** de **17 264** casos reales (55.1 %) y asigna erróneamente **5 333** a la clase 0 y **2 428** a la clase 2. En la categoría de fallecimiento, de **66** ocurrencias reales el modelo acierta **34** (51.5 %), dejando **19** y **13** instancias en las clases 0 y 1, respectivamente.  

La curva ROC micro-average apenas supera la diagonal de azar, y el AUC macro de **0.4753** confirma que el BalancedBaggingClassifier no logra discriminar con fiabilidad las tres clases cuando se evalúa en conjunto. En el resumen de métricas, el F1-score macro es de **0.3679**, la precisión macro alcanza **0.6039** y la exactitud global es **0.6039**, valores claramente inferiores a los de XGBoost (F1 ≈ 0.4475) y Random Forest (F1 ≈ 0.4332).  

En resumen, aunque este enfoque reduce drásticamente el tiempo de entrenamiento, su capacidad para identificar correctamente los desenlaces graves (especialmente la hospitalización frente a la muerte) es limitada, lo que lo sitúa por debajo de los modelos de referencia en términos de F1 macro y AUC.  


```{note}
Además, se probaron otros métodos como **RUSBoostClassifier**, **Balanced Random Forests**, **CatBoost**, **CusBoost**, **SMOTEBoost**, entre otros. Sin embargo, todos ellos mostraron un rendimiento aún más deficiente en F1-score macro y AUC, por lo que se decidió no incluir sus resultados detallados en este análisis.  
```

### **Conclusión Final y Comparación de Modelos**

En esta última etapa hemos comparado nuestro modelo de referencia —**XGBoost (binario)**— con los tres enfoques alternativos evaluados (HistGradientBoostingClassifier, EasyEnsembleClassifier y BalancedBaggingClassifier). A continuación se muestra un resumen de sus métricas clave:

| Modelo                           | F1 (Macro) | AUC (Macro) | CPU Time (s) |
|:---------------------------------|:----------:|:-----------:|:------------:|
| **XGBoost (binario)**            |   **0.6792**   |   **0.8143**    |     46.63     |
| HistGradientBoostingClassifier   |    0.4474   |    0.7505    |    296.07     |
| EasyEnsembleClassifier           |    0.3688   |    0.6849    |    597.09     |
| BalancedBaggingClassifier        |    0.3679   |    0.4753    |     0.36      |

Tal como se observa, **XGBoost** mantiene una ventaja clara en **F1-score macro** y **AUC macro**, a la vez que ofrece un tiempo de cómputo competitivo. Ninguno de los métodos alternativos logró superar su desempeño, por lo que se confirma su idoneidad para la detección de víctimas en riesgo en este contexto.  


### **Motivación para la fusión de clases**

El objetivo principal de esta transformación es mejorar la capacidad de los modelos para identificar con precisión a las víctimas en peligro —ya sea por hospitalización o fallecimiento— en un contexto donde la categoría de mayor interés (clase 2: víctima fallecida) constituye apenas el 0.07 % de los registros. A pesar de haber aplicado SMOTE y validación cruzada estratificada, ambos algoritmos de referencia —XGBoost clásico y Random Forest— presentan dificultades significativas para clasificar correctamente esta clase minoritaria.  

En la matriz de confusión de XGBoost se observa un elevado número de falsos negativos para la clase 2, y en la de Random Forest  ocurre un patrón similar, lo que implica que gran parte de los casos de fallecimiento quedan ocultos en las predicciones de hospitalización o incluso en las de ausencia de gravedad. Este marcado sesgo y la extrema rareza de la etiqueta “fallecimiento” reducen drásticamente la efectividad del F1-score macro y comprometen la detección de los casos más críticos.

Para mitigar este problema, se decidió agrupar las dos categorías de desenlace grave (hospitalización y muerte) en una sola etiqueta de “víctima en peligro” frente a “víctima no grave”. Esta unificación convierte el problema original en un escenario de clasificación binaria, lo cual:


*   Incrementa la representación de la clase de interés, facilitando el aprendizaje de patrones asociados a desenlaces graves.
*   Permite aplicar algoritmos más sensibles a la detección de casos críticos y optimizar directamente la métrica objetivo (F1-score macro).
*   Reduce el riesgo de que los casos de fallecimiento queden diluidos o pasen inadvertidos en un esquema multiclase con muy baja frecuencia.

En la sección se detalla el proceso completo de reconstrucción de etiquetas, la aplicación de SMOTE sobre cada fold y la evaluación de los modelos binarios, analizando en profundidad los resultados obtenidos en términos de F1-score macro, AUC y tiempo de cómputo.  


## **Cambio a clasificación binaria**

### **Creación de la variable `Victima_Peligro` y reentrenamiento**

Para mejorar la capacidad de los modelos de identificar a las víctimas en situación de riesgo (hospitalización o muerte), se redefinió la variable objetivo original **Desenlace_Grave** (con tres categorías) en una nueva variable binaria llamada **Victima_Peligro**. La transformación se realizó de la siguiente forma:

- **Si Desenlace_Grave = 0** (víctima viva y no hospitalizada) → **Victima_Peligro = 0**  
- **Si Desenlace_Grave = 1** (víctima viva y hospitalizada) o **2** (víctima fallecida) → **Victima_Peligro = 1**

Una vez creada esta variable, se guardó el nuevo dataset y se repitió el proceso de validación cruzada estratificada con 5 folds, aplicando SMOTE sobre cada conjunto de entrenamiento para balancear la proporción de clases.



#### **Stratified Cross Validation**

In [ ]:
# Separación de características y variable objetivo
X = Data.drop(columns=["Victima_Peligro", "Desenlace_Grave"])
y = Data['Victima_Peligro']

# StratifiedKFold con 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
fold = 1

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    train_shape = X_train.shape
    test_shape = X_test.shape

    # Distribución de clases en cada conjunto
    train_dist = y_train.value_counts(normalize=True).sort_index()
    test_dist = y_test.value_counts(normalize=True).sort_index()

    fold_results.append({
        "Fold": fold,
        "Train Shape": train_shape,
        "Test Shape": test_shape,
        "Train Distribution": train_dist,
        "Test Distribution": test_dist
    })
    fold += 1

for result in fold_results:
    display(HTML(f"<h3>Fold {result['Fold']}</h3>"))
    display(HTML(f"<b>Train shape:</b> {result['Train Shape']}<br>"
                 f"<b>Test shape:</b> {result['Test Shape']}"))
    dist_df = pd.DataFrame({
        "Train Distribution": result['Train Distribution'],
        "Test Distribution": result['Test Distribution']
    })
    dist_df = dist_df * 100
    dist_df = dist_df.round(2)
    display(dist_df)
    display(HTML("<hr>"))

,Train Distribution,Test Distribution
Victima_Peligro,,
0,82.97,82.97
1,17.03,17.03


,Train Distribution,Test Distribution
Victima_Peligro,,
0,82.97,82.97
1,17.03,17.03


,Train Distribution,Test Distribution
Victima_Peligro,,
0,82.97,82.97
1,17.03,17.03


,Train Distribution,Test Distribution
Victima_Peligro,,
0,82.97,82.97
1,17.03,17.03


,Train Distribution,Test Distribution
Victima_Peligro,,
0,82.97,82.97
1,17.03,17.03


#### **Aplicación de SMOTE**

In [ ]:
# Distribución original
display(HTML("<h3>Distribución de clases original (entrenamiento)</h3>"))
dist_orig = pd.DataFrame(y_train.value_counts(normalize=True).sort_index())
dist_orig.columns = ['Proporción']
display(dist_orig.style.format({'Proporción': "{:.2%}"}))

# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
display(HTML("<h3>Distribución de clases después de SMOTE</h3>"))
dist_smote = pd.DataFrame(pd.Series(y_train_smote).value_counts(normalize=True).sort_index())
dist_smote.columns = ['Proporción']
display(dist_smote.style.format({'Proporción': "{:.2%}"}))

,Proporción
Victima_Peligro,
0,82.97%
1,17.03%


,Proporción
Victima_Peligro,
0,50.00%
1,50.00%


En las cinco particiones de la validación cruzada estratificada, la proporción de la clase `Victima_Peligro` se mantuvo constante: aproximadamente **82.97 %** de los ejemplos correspondieron a la clase 0 (víctima no grave) y **17.03 %** a la clase 1 (víctima en peligro), tanto en los conjuntos de entrenamiento como en los de prueba. Esto confirma que la estratificación preservó el desbalance original en cada fold.

Al aplicar SMOTE únicamente sobre los datos de entrenamiento en cada pliegue, se logró un balance perfecto de las clases: **50.00 %** de ejemplos en clase 0 y **50.00 %** en clase 1. De este modo, garantizamos que el modelo vea un número equiparable de muestras de víctimas en riesgo y no riesgo durante el entrenamiento, mejorando su capacidad para aprender a distinguir ambas categorías.


## **Reentrenamiento de modelos con mejor rendimiento**

### **XGBoost (binario)**

Para aprovechar la ventaja que mostró XGBoost en el esquema multiclase, a continuación aplicamos el mismo enfoque sobre la variable **Victima_Peligro** (dos clases). Se construyó un pipeline que utiliza un escalado estándar y un `XGBClassifier` configurado para `objective="binary:logistic"`, y se optimizaron los hiperparámetros `n_estimators` y `max_depth` mediante `GridSearchCV` con 5 folds y scoring `f1_macro`. Este proceso permite afinar el modelo tanto para maximizar la detección de víctimas en riesgo como para controlar el tiempo de cómputo.

Una vez obtenido el mejor estimador, se evalúa su rendimiento en el conjunto de prueba a través de la matriz de confusión, la curva ROC y un resumen de métricas (Precision, Recall y F1-score macro, AUC y tiempo de entrenamiento). Además, incorporamos **LIME** para generar explicaciones locales de las predicciones, de modo que se identifiquen las variables que más contribuyen a clasificar cada caso como “No Peligro” o “Peligro”.


In [ ]:
# Pipeline XGBoost + GridSearchCV
memory = Memory("./cache_dir", verbose=0)

xgb_pipeline = Pipeline([
    ("scaler", StandardScaler(copy=False)),
    ("xgb", XGBClassifier(
        objective="binary:logistic",
        n_estimators=200,
        n_jobs=-1,
        random_state=42,
        tree_method="auto",
        use_label_encoder=False,
        eval_metric="logloss"
    ))
], memory=memory)

param_grid = {
    "xgb__n_estimators": [100, 200],
    "xgb__max_depth":    [3, 6]
}

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    error_score="raise"
)

# Entrenamiento
start_time = time.process_time()
grid_search.fit(X_train_smote, y_train_smote)
training_time = time.process_time() - start_time

best_model = grid_search.best_estimator_
print("Mejores parámetros encontrados:", grid_search.best_params_)

# Predicciones y métricas
y_pred       = best_model.predict(X_test)
y_proba_pos  = best_model.predict_proba(X_test)[:, 1]

report       = classification_report(y_test, y_pred, output_dict=True)
accuracy_glob = report["accuracy"]
roc_auc      = roc_auc_score(y_test, y_proba_pos)

# Matriz de Confusión
cm     = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))
z_text = [[str(v) for v in row] for row in cm]

fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labels],
    y=[f"True_{l}" for l in labels],
    annotation_text=z_text,
    colorscale="Purples"
)
fig_cm.update_layout(
    title_text="Matriz de Confusión – XGBoost",
    xaxis=dict(title="Predicción"),
    yaxis=dict(title="Valor Real"),
    font=dict(family="Inter")
)
fig_cm.show()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_pos)
fig_roc = go.Figure([
    go.Scatter(x=fpr, y=tpr, mode="lines",
               name=f"ROC (AUC={roc_auc:.4f})",
               line=dict(color="darkmagenta", width=3)),
    go.Scatter(x=[0,1], y=[0,1], mode="lines",
               name="Azar", line=dict(color="gray", dash="dash"))
])
fig_roc.update_layout(
    title="Curva ROC – XGBoost",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Inter", size=13, color="black"),
    width=700, height=500
)
fig_roc.show()

# Resumen de métricas
summary_data = {
    "Precision (Macro)": [report["macro avg"]["precision"]],
    "Recall (Macro)"   : [report["macro avg"]["recall"]],
    "F1-score (Macro)" : [report["macro avg"]["f1-score"]],
    "Accuracy"         : [accuracy_glob],
    "AUC"              : [roc_auc],
    "Training Time (s)": [training_time]
}
summary_df = pd.DataFrame(summary_data)
display(HTML("<h2>Resumen de Métricas – XGBoost</h2>"))
display(summary_df.style.format("{:.4f}"))

# LIME
explainer = LimeTabularExplainer(
    training_data=X_train_smote.values,
    feature_names=X_train_smote.columns.tolist(),
    class_names=["No Peligro", "Peligro"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

idx = 0
exp = explainer.explain_instance(
    data_row=X_test.values[idx],
    predict_fn=best_model.predict_proba,
    num_features=5
)

lime_list = exp.as_list()
features, weights = zip(*lime_list)
colors = ['#6A0DAD' if w > 0 else '#DDA0DD' for w in weights]

fig_lime = go.Figure(go.Bar(
    x=list(weights),
    y=list(features),
    orientation='h',
    marker=dict(color=colors)
))
fig_lime.update_layout(
    title_text="Explicación LIME – XGBoost",
    xaxis_title="Peso de la característica",
    yaxis_title="Características",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Inter", color="black"),
    width=700, height=500
)
fig_lime.show()


Mejores parámetros encontrados: {'xgb__max_depth': 6, 'xgb__n_estimators': 200}


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC,Training Time (s)
0,0.7232,0.6566,0.6792,0.8446,0.8143,46.6343


La matriz de confusión revela que el modelo identifica casi la totalidad de los casos no graves (alrededor del 94 % de verdaderos negativos) y mantiene un nivel bajo de falsos positivos, pero presenta dificultades para detectar víctimas en riesgo: solo identifica alrededor del 37 % de los verdaderos positivos, dejando sin clasificar más de la mitad de los casos críticos. La curva ROC, con un AUC próximo a 0.82, indica que, a pesar de este déficit en recall para la clase de interés, el clasificador dispone de un buen poder de discriminación global; ajustando el umbral se podría mejorar la sensibilidad a costa de más falsos positivos. En conjunto, el F1-score macro (cercano a 0.68) refleja un compromiso razonable entre precisión y recall, aunque aún deja margen de mejora en la detección de “Víctima en peligro”.

A continuación, se muestran las cinco variables que más influyeron en la predicción según LIME para un caso de prueba, con sus pesos y la dirección de su efecto sobre la probabilidad de riesgo:

| Característica                          | Peso    | Influencia     |
|-----------------------------------------|--------:|----------------|
| `TIP_SS_I <= 0.00`                      | +0.035  | Aumenta riesgo |
| `GP_OTROS <= 1.00`                      | +0.025  | Aumenta riesgo |
| `Diferencia_ABS_NOT_CON <= 0.00`        | +0.015  | Aumenta riesgo |
| `estrato > 1.78`                        | +0.010  | Aumenta riesgo |
| `EDAD > 29.00`                          | –0.020  | Disminuye riesgo |


### **Random Forest (binario)**

Ahora evaluamos el segundo mejor modelo, **Random Forest**, sobre la variable binaria **Victima_Peligro** para comprobar cómo se comporta en la detección de víctimas en riesgo. Siguiendo el mismo esquema que con XGBoost, escalamos las características, entrenamos un `RandomForestClassifier` con `class_weight="balanced"` para corregir el desbalance y medimos su desempeño mediante validación cruzada estratificada, matriz de confusión, curva ROC y métricas clave (F1-score macro, AUC y tiempo de entrenamiento). Además, incorporamos **LIME** para generar explicaciones locales que faciliten la interpretación de las predicciones y ayuden a identificar las variables que más influyen en la clasificación de cada caso.  


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd
import numpy as np
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
from lime.lime_tabular import LimeTabularExplainer
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import display, HTML

# 1) Definir caché y pipeline
memory = Memory("./cache_dir", verbose=0)
rf_pipeline = Pipeline([
    ("scaler", StandardScaler(copy=False)),
    ("rf", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        n_estimators=100,
        max_depth=None,
        class_weight="balanced"
    ))
], memory=memory)

# Entrenamiento
start_time = time.process_time()
rf_pipeline.fit(X_train_smote, y_train_smote)
training_time = time.process_time() - start_time

# Predicciones y probabilidades de la clase positiva
y_pred     = rf_pipeline.predict(X_test)
y_proba_pos = rf_pipeline.predict_proba(X_test)[:, 1]

# Reporte de clasificación
report = classification_report(y_test, y_pred, output_dict=True)
accuracy_global = report["accuracy"]
display(HTML("<h2>Classification Report – Random Forest</h2>"))
display(pd.DataFrame(report).transpose().style.format("{:.4f}"))

# AUC binario
roc_auc = roc_auc_score(y_test, y_proba_pos)

# Matriz de confusión
cm     = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))
z_text = [[str(v) for v in row] for row in cm]
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labels],
    y=[f"True_{l}" for l in labels],
    annotation_text=z_text,
    colorscale="Purples"
)
fig_cm.update_layout(
    title_text="Matriz de Confusión – Random Forest",
    xaxis_title="Predicción",
    yaxis_title="Valor Real",
    font=dict(family="Inter")
)
fig_cm.show()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_pos)
fig_roc = go.Figure([
    go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"AUC={roc_auc:.4f}",
        line=dict(color="darkmagenta", width=3)
    ),
    go.Scatter(
        x=[0,1], y=[0,1], mode="lines",
        name="Azar", line=dict(color="gray", dash="dash")
    )
])
fig_roc.update_layout(
    title="Curva ROC – Random Forest",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter", size=13, color="black"),
    width=700, height=500
)
fig_roc.show()

# Resumen de métricas
summary_data = {
    "Precision (Macro)": [report["macro avg"]["precision"]],
    "Recall (Macro)"   : [report["macro avg"]["recall"]],
    "F1-score (Macro)" : [report["macro avg"]["f1-score"]],
    "Accuracy"         : [accuracy_global],
    "AUC"              : [roc_auc],
    "Training Time (s)": [training_time]
}
summary_df = pd.DataFrame(summary_data)
display(HTML("<h2>Resumen de Métricas – Random Forest</h2>"))
display(summary_df.style.format("{:.4f}"))

# Importancia de características
importances    = rf_pipeline.named_steps["rf"].feature_importances_
feature_names  = X_train_smote.columns
importance_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

fig_imp = go.Figure(go.Bar(
    x=importance_df["Feature"],
    y=importance_df["Importance"],
    marker_color="darkmagenta"
))
fig_imp.update_layout(
    title="Importancia de Características – Random Forest",
    xaxis_title="Características",
    yaxis_title="Importancia",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter", size=13, color="black")
)
fig_imp.show()

# LIME
explainer = LimeTabularExplainer(
    training_data=X_train_smote.values,
    feature_names=X_train_smote.columns.tolist(),
    class_names=["No Peligro", "Peligro"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

idx = 0
exp = explainer.explain_instance(
    data_row=X_test.values[idx],
    predict_fn=rf_pipeline.predict_proba,
    num_features=5
)

features, weights = zip(*exp.as_list())
colors = ['#6A0DAD' if w > 0 else '#DDA0DD' for w in weights]

fig_lime = go.Figure(go.Bar(
    x=list(weights),
    y=list(features),
    orientation="h",
    marker=dict(color=colors)
))
fig_lime.update_layout(
    title_text="Explicación LIME – Random Forest",
    xaxis_title="Peso de la característica",
    yaxis_title="Características",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter", color="black"),
    width=700, height=500
)
fig_lime.show()


,precision,recall,f1-score,support
0,0.8777,0.9283,0.9023,84408.0000
1,0.5143,0.3699,0.4303,17330.0000
accuracy,0.8332,0.8332,0.8332,0.8332
macro avg,0.6960,0.6491,0.6663,101738.0000
weighted avg,0.8158,0.8332,0.8219,101738.0000


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC,Training Time (s)
0,0.6960,0.6491,0.6663,0.8332,0.7872,193.9913


El Random Forest binario confirma su solidez en la detección de víctimas en riesgo: de 84 408 casos no graves, acierta 78 353 (≈92.8 %) y genera 6 055 falsos positivos; en la clase de interés, identifica 10 919 de 17 330 víctimas en peligro (≈63.0 %) y pasa por alto 6 411 (37.0 %) falsos negativos. La curva ROC, con un AUC de ≈0.7872, muestra una buena capacidad de discriminación, aunque algo inferior a XGBoost. El F1‐score macro de ≈0.6663 refleja un equilibrio razonable entre precisión y recall, manteniéndose competitivo a pesar de un tiempo de cómputo mayor (~194 s).  

A continuación, las cinco variables más influyentes según LIME para una instancia de prueba, con sus pesos y dirección de efecto:

| Característica                   | Peso    | Influencia     |
|----------------------------------|--------:|----------------|
| `TIP_SS_C <= 1.00`               | –0.100  | Disminuye riesgo |
| `estrato` (1.78–2.00)            | +0.120  | Aumenta riesgo |
| `DiferenciaAnios <= 0.00`        | +0.080  | Aumenta riesgo |
| `TIP_SS_S <= 0.00`               | +0.060  | Aumenta riesgo |
| `Diferencia_ABS_NOT_CON <= 0.00` | +0.040  | Aumenta riesgo |


## **Comparación con Otros Modelos en Clasificación Binaria**

Además de XGBoost y Random Forest, se evaluaron tres enfoques adicionales para la detección de víctimas en riesgo en el escenario de dos clases (`Victima_Peligro`). El objetivo fue comprobar si alguna de estas técnicas logra mejorar el F1-score macro, o bien mantenerlo similar mientras reduce el tiempo de cómputo. Los modelos considerados fueron:

1. **Balanced Random Forest (BRF)**: un ensamble de árboles que aplica undersampling interno en cada estimador para equilibrar automáticamente las clases.  
2. **HistGradientBoostingClassifier con early stopping**: la misma implementación basada en histogramas, ahora en su versión binaria y detenida cuando la métrica deja de mejorar.  
3. **Pipeline SMOTE → StandardScaler → AdaBoost** con RandomizedSearchCV: oversampling inicial sobre la clase minoritaria combinado con búsqueda aleatoria de hiperparámetros para AdaBoost.

A continuación se presenta, para cada método, una breve introducción de su funcionamiento seguido del código utilizado y la interpretación de los resultados obtenidos.  


### **Balanced Random Forest (BRF)**

En esta sección aplicamos **Balanced Random Forest** al problema binario de `Victima_Peligro`. Este enfoque extiende el Random Forest tradicional introduciendo un muestreo interno de la clase mayoritaria en cada árbol, de modo que todos los estimadores se entrenan con subconjuntos balanceados. Para optimizar su rendimiento, se construyó un pipeline que incluye escalado de características y un `BalancedRandomForestClassifier` cuyos hiperparámetros `n_estimators` y `max_depth` se ajustaron mediante `GridSearchCV` con scoring `f1_macro` y validación cruzada de 5 folds. Tras seleccionar el mejor modelo, evaluamos su desempeño en el conjunto de prueba mediante matriz de confusión, curva ROC, métricas clave (precisión, recall, F1-score y AUC) y tiempo de entrenamiento. Finalmente, utilizamos **LIME** para generar explicaciones locales que identifiquen las cinco variables más influyentes en la predicción de riesgo.  


In [ ]:
# --- Pipeline con Balanced Random Forest ---
memory = Memory("./cache_dir", verbose=0)

brf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("brf", BalancedRandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
], memory=memory)

param_grid = {
    "brf__n_estimators": [100, 200],
    "brf__max_depth"  : [3, 6]
}

grid_search = GridSearchCV(
    estimator=brf_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    error_score="raise"
)

start = time.process_time()
grid_search.fit(X_train, y_train)
train_time = time.process_time() - start

best_model = grid_search.best_estimator_
print("Mejores parámetros BRF:", grid_search.best_params_)

# Evaluación en X_test,y_test ---
y_pred      = best_model.predict(X_test)
y_proba_pos = best_model.predict_proba(X_test)[:,1]

report = classification_report(y_test, y_pred, output_dict=True)
acc    = report["accuracy"]
auc    = roc_auc_score(y_test, y_proba_pos)

# Matriz de confusión
cm     = confusion_matrix(y_test, y_pred)
labs   = sorted(np.unique(y_test))
z_txt  = [[str(v) for v in row] for row in cm]
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labs],
    y=[f"True_{l}" for l in labs],
    annotation_text=z_txt,
    colorscale="Purples"
)
fig_cm.update_layout(
    title_text="Matriz de Confusión – BRF",
    xaxis_title="Predicción",
    yaxis_title="Valor Real",
    font=dict(family="Inter")
)
fig_cm.show()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_pos)
fig_roc = go.Figure([
    go.Scatter(x=fpr, y=tpr, mode="lines",
               name=f"AUC={auc:.4f}", line=dict(color="darkmagenta", width=3)),
    go.Scatter(x=[0,1], y=[0,1], mode="lines",
               name="Azar", line=dict(color="gray", dash="dash"))
])
fig_roc.update_layout(
    title="Curva ROC – BRF",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Inter"), width=700, height=500
)
fig_roc.show()

# Resumen de métricas
summary = {
    "Precision (Macro)": [report["macro avg"]["precision"]],
    "Recall (Macro)"   : [report["macro avg"]["recall"]],
    "F1-score (Macro)" : [report["macro avg"]["f1-score"]],
    "Accuracy"         : [acc],
    "AUC"              : [auc],
    "Training Time (s)": [train_time]
}
df_sum = pd.DataFrame(summary)
display(HTML("<h2>Resumen de Métricas – BRF</h2>"))
display(df_sum.style.format("{:.4f}"))

# LIME
explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=["No Peligro","Peligro"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

idx = 0
exp = explainer.explain_instance(
    data_row=X_test.values[idx],
    predict_fn=best_model.predict_proba,
    num_features=5
)

feat_wts = exp.as_list()
feats, wts  = zip(*feat_wts)
cols = ['#6A0DAD' if w>0 else '#DDA0DD' for w in wts]

fig_lime = go.Figure(go.Bar(
    x=list(wts), y=list(feats), orientation='h',
    marker=dict(color=cols)
))
fig_lime.update_layout(
    title_text="LIME – BRF",
    xaxis_title="Peso de la característica",
    yaxis_title="Características",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Inter"), width=700, height=500
)
fig_lime.show()


Mejores parámetros BRF: {'brf__max_depth': 6, 'brf__n_estimators': 200}


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC,Training Time (s)
0,0.6070,0.6854,0.5850,0.6516,0.7515,45.1627


Los mejores parámetros encontrados (`max_depth=6`, `n_estimators=200`) permitieron que el Balanced Random Forest completara su entrenamiento en **45.16 s**, un tiempo moderado dentro de las alternativas evaluadas.

La matriz de confusión muestra un claro sesgo hacia la clase mayoritaria: de **84 408** ejemplos no graves (Victima_Peligro=0), el modelo acierta **53 529** (≈63.5 %) y confunde **30 879** como en riesgo. En la clase de interés, identifica **4 565** verdaderos positivos de **17 330** casos (≈26.3 %) y deja **12 765** falsos negativos sin detectar. Esto indica que, aunque el BRF conserva una precisión aceptable para la clase mayoritaria, su capacidad para reconocer a las víctimas en peligro es limitada.

La curva ROC (AUC ≈ 0.7515) refleja una discriminación moderada entre ambas categorías, ligeramente inferior a otros modelos de boosting, y su F1-score macro de **0.5850** confirma un equilibrio aún mejorable entre precisión y recall en el conjunto binario.

A continuación, las cinco variables más influyentes según LIME para una instancia de prueba, con sus pesos y la dirección de su efecto sobre la probabilidad de riesgo:

| Característica               | Peso    | Influencia       |
|------------------------------|--------:|------------------|
| `22.00 < EDAD <= 33.00`      | –0.080  | Disminuye riesgo |
| `UNI_MED <= 1.00`            | –0.060  | Disminuye riesgo |
| `COD_MUN_R > 473.00`         | –0.040  | Disminuye riesgo |
| `Diferencia_ABS_NOT_CON <= 0`| –0.020  | Disminuye riesgo |
| `0.00 < TIP_SS_C <= 1.00`    | –0.010  | Disminuye riesgo |


### **HistGradientBoostingClassifier (binario)**

A continuación aplicamos la versión binaria de **HistGradientBoostingClassifier** con early stopping para optimizar automáticamente el número de iteraciones y `class_weight="balanced"` para penalizar la clase mayoritaria. Se fijó una tasa de aprendizaje moderada (`learning_rate=0.1`) y se reservó el 10 % del conjunto de entrenamiento como validación interna, deteniendo el entrenamiento si no hay mejora tras 10 rondas consecutivas. Este enfoque busca combinar la eficiencia del método de histogramas con un control de sobreajuste y un ajuste de desequilibrio de clases, con el fin de maximizar el F1-score macro sin incurrir en largos tiempos de cómputo. A continuación presentamos el código de entrenamiento, seguido de la interpretación de sus resultados.  


In [ ]:
# Pipeline con HistGB + early stopping
hgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("hgb", HistGradientBoostingClassifier(
        class_weight="balanced",       # penaliza la clase mayoritaria
        learning_rate=0.1,
        max_iter=200,                  # número máximo de iteraciones
        early_stopping="auto",         # para cuando no mejore la loss
        validation_fraction=0.1,       # 10% del train como validación interna
        n_iter_no_change=10,           # 10 rondas sin mejora → stop
        random_state=42
    ))
])

# Entrenamiento
start = time.process_time()
hgb_pipeline.fit(X_train_smote, y_train_smote)
train_time = time.process_time() - start
print(f"✔️   Entrenado en {train_time:.2f}s")

# Predicción y métricas sobre X_test / y_test
y_pred     = hgb_pipeline.predict(X_test)
y_proba    = hgb_pipeline.predict_proba(X_test)[:,1]

# Classification report
rep = classification_report(y_test, y_pred, output_dict=True)
display(HTML("<h2>Classification Report – HistGB</h2>"))
display(pd.DataFrame(rep).transpose().style.format("{:.4f}"))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
labs = sorted(np.unique(y_test))
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labs],
    y=[f"True_{l}" for l in labs],
    annotation_text=[[str(v) for v in row] for row in cm],
    colorscale="Purples"
)
fig_cm.update_layout(
    title_text="Matriz de Confusión – HistGB",
    xaxis_title="Predicción", yaxis_title="Valor Real",
    font=dict(family="Inter")
)
fig_cm.show()

# Curva ROC
auc_ = roc_auc_score(y_test, y_proba)
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig_roc = go.Figure([
    go.Scatter(x=fpr, y=tpr, mode="lines",
               name=f"AUC={auc_:.4f}", line=dict(color="darkmagenta", width=3)),
    go.Scatter(x=[0,1], y=[0,1], mode="lines",
               name="Azar", line=dict(color="gray", dash="dash"))
])
fig_roc.update_layout(
    title="Curva ROC – HistGB",
    xaxis_title="False Positive Rate", yaxis_title="True Positive Rate",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Inter"), width=700, height=500
)
fig_roc.show()

# Resumen de métricas
summary = {
    "Precision (Macro)": [rep["macro avg"]["precision"]],
    "Recall (Macro)"   : [rep["macro avg"]["recall"]],
    "F1-score (Macro)" : [rep["macro avg"]["f1-score"]],
    "Accuracy"         : [rep["accuracy"]],
    "AUC"              : [auc_],
    "Train Time (s)"   : [train_time]
}
display(HTML("<h2>Resumen de Métricas – HistGradientBoosting</h2>"))
display(pd.DataFrame(summary).style.format("{:.4f}"))

# Explicación LIME
explainer = LimeTabularExplainer(
    training_data=X_train_smote.values,
    feature_names=X_train_smote.columns.tolist(),
    class_names=["No Peligro","Peligro"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

exp = explainer.explain_instance(
    data_row=X_test.values[0],
    predict_fn=hgb_pipeline.predict_proba,
    num_features=5
)
feats, wts = zip(*exp.as_list())
cols = ['#6A0DAD' if w>0 else '#DDA0DD' for w in wts]

fig_lime = go.Figure(go.Bar(
    x=list(wts), y=list(feats),
    orientation='h',
    marker=dict(color=cols)
))
fig_lime.update_layout(
    title_text="Explicación LIME – HistGB",
    xaxis_title="Peso de la característica",
    yaxis_title="Características",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family="Inter"), width=700, height=500
)
fig_lime.show()


✔️   Entrenado en 73.24s


,precision,recall,f1-score,support
0,0.8748,0.9421,0.9072,84408.0000
1,0.5488,0.3430,0.4222,17330.0000
accuracy,0.8401,0.8401,0.8401,0.8401
macro avg,0.7118,0.6426,0.6647,101738.0000
weighted avg,0.8192,0.8401,0.8246,101738.0000


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC,Train Time (s)
0,0.7118,0.6426,0.6647,0.8401,0.8014,73.2393


El HistGradientBoosting binario mostró un desempeño intermedio: de los **84 408** casos “no riesgo” predice correctamente **79 522** (≈94.2 %) y marca **4 886** falsos positivos; en la clase “riesgo” detecta **11 386** de **17 330** verdaderos positivos (≈65.7 %) y deja **5 944** falsos negativos. Este balance se refleja en un **F1-score macro de 0.6647** y un **AUC de 0.8014**, con una exactitud global de **0.8401**. El tiempo de entrenamiento fue de **73.24 s**, gracias al early stopping, logrando un buen compromiso entre calidad de predicción y coste computacional.

A continuación, las cinco variables más influyentes según LIME para una instancia de prueba:

| Característica                          | Peso    | Influencia       |
|-----------------------------------------|--------:|------------------|
| `0.00 < TIP_SS_C <= 1.00`               | –0.100  | Disminuye riesgo |
| `TIP_SS_S <= 0.00`                      | +0.080  | Aumenta riesgo   |
| `EDAD > 29.00`                          | –0.060  | Disminuye riesgo |
| `1.78 < estrato <= 2.00`                | +0.060  | Aumenta riesgo   |
| `Diferencia_ABS_NOT_CON <= 0.00`        | +0.030  | Aumenta riesgo   |


### **SMOTE + AdaBoost (RandomizedSearchCV)**

Para completar la comparativa en binario, implementamos un pipeline que integra SMOTE previo al escalado de características y un **AdaBoostClassifier**, optimizado mediante **RandomizedSearchCV** con solo cuatro combinaciones de hiperparámetros (`n_estimators` y `learning_rate`) y validación en tres folds. Este enfoque permite reforzar la clase minoritaria en cada iteración y explorar rápidamente distintas configuraciones, con el objetivo de mejorar el F1-score macro sin incurrir en largos tiempos de búsqueda. Posteriormente, evaluamos el modelo resultante con matriz de confusión, curva ROC, métricas clave y explicabilidad local con LIME en tonos morados para facilitar la interpretación de las variables más determinantes en la clasificación de “Peligro” versus “No Peligro”.  


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd
import numpy as np
from joblib import Memory
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from imblearn.over_sampling import SMOTE
from lime.lime_tabular import LimeTabularExplainer
import plotly.figure_factory as ff
import plotly.graph_objects as go
from IPython.display import display, HTML

# 1) Pipeline SMOTE → Scaling → AdaBoost
memory = Memory("./cache_dir", verbose=0)
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("ada", AdaBoostClassifier(random_state=42))
], memory=memory)

# 2) Distribución de parámetros
param_dist = {
    "ada__n_estimators":  [50, 100, 200],
    "ada__learning_rate": [0.5, 1.0]
}

# 3) RandomizedSearchCV para acelerar
rand_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=4,            # solo 4 combinaciones
    scoring="f1_macro",
    cv=3,                # 3 folds
    n_jobs=-1,
    verbose=2,
    random_state=42,
    error_score="raise"
)

# 4) Entrenamiento
start_time = time.process_time()
rand_search.fit(X_train_smote, y_train_smote)
training_time = time.process_time() - start_time

best_model = rand_search.best_estimator_
print("Mejores parámetros encontrados:", rand_search.best_params_)

# 5) Predicciones
y_pred      = best_model.predict(X_test)
y_proba_pos = best_model.predict_proba(X_test)[:, 1]

# 6) Reporte de clasificación
report = classification_report(y_test, y_pred, output_dict=True)

# 7) Métricas
accuracy_global = report["accuracy"]
roc_auc = roc_auc_score(y_test, y_proba_pos)

# 8) Matriz de Confusión
cm     = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))
z_text = [[str(v) for v in row] for row in cm]

fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Pred_{l}" for l in labels],
    y=[f"True_{l}" for l in labels],
    annotation_text=z_text,
    colorscale="Purples"
)
fig_cm.update_layout(
    title_text="Matriz de Confusión – SMOTE + AdaBoost",
    xaxis=dict(title="Predicción"),
    yaxis=dict(title="Valor Real"),
    font=dict(family="Inter")
)
fig_cm.show()

# 9) Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_pos)
fig_roc = go.Figure([
    go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"AUC={roc_auc:.4f}", line=dict(color="darkmagenta", width=3)
    ),
    go.Scatter(
        x=[0,1], y=[0,1], mode="lines",
        name="Azar", line=dict(color="gray", dash="dash")
    )
])
fig_roc.update_layout(
    title="Curva ROC – SMOTE + AdaBoost",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter", size=13, color="black"),
    width=700, height=500
)
fig_roc.show()

# 10) Resumen de métricas
summary_data = {
    "Precision (Macro)": [report["macro avg"]["precision"]],
    "Recall (Macro)"   : [report["macro avg"]["recall"]],
    "F1-score (Macro)" : [report["macro avg"]["f1-score"]],
    "Accuracy"         : [accuracy_global],
    "AUC"              : [roc_auc],
    "Training Time (s)": [training_time]
}
summary_df = pd.DataFrame(summary_data)
display(HTML("<h2>Resumen de Métricas – SMOTE + AdaBoost (RandomizedSearchCV)</h2>"))
display(summary_df.style.format("{:.4f}"))

# 11) LIME en tonos morados
explainer = LimeTabularExplainer(
    training_data=X_train_smote.values,
    feature_names=X_train_smote.columns.tolist(),
    class_names=["No Peligro", "Peligro"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

idx = 0
exp = explainer.explain_instance(
    data_row=X_test.values[idx],
    predict_fn=best_model.predict_proba,
    num_features=5
)

features, weights = zip(*exp.as_list())
colors = ['#6A0DAD' if w > 0 else '#DDA0DD' for w in weights]

fig_lime = go.Figure(go.Bar(
    x=list(weights),
    y=list(features),
    orientation='h',
    marker=dict(color=colors)
))
fig_lime.update_layout(
    title_text="Explicación LIME – SMOTE + AdaBoost",
    xaxis_title="Peso de la característica",
    yaxis_title="Características",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter", color="black"),
    width=700, height=500
)
fig_lime.show()


Fitting 3 folds for each of 4 candidates, totalling 12 fits
Mejores parámetros encontrados: {'ada__n_estimators': 50, 'ada__learning_rate': 0.5}


,Precision (Macro),Recall (Macro),F1-score (Macro),Accuracy,AUC,Training Time (s)
0,0.5867,0.6348,0.5858,0.6909,0.6973,29.9179


El pipeline de SMOTE seguido de AdaBoost, optimizado con RandomizedSearchCV, completó su entrenamiento en **29.92 s**. La matriz de confusión muestra que de **84 408** casos no graves el modelo acierta **60 764** (≈72.0 %) y produce **23 644** falsos positivos, mientras que en la clase de interés detecta **7 804** de **17 330** víctimas en riesgo (≈45.0 %) y deja **9 526** falsos negativos. Esto se traduce en un **F1-score macro de 0.5858**, un **recall macro de 0.6348** y una **precisión macro de 0.5867**, reflejando un equilibrio moderado entre sensibilidad y exactitud. La curva ROC alcanza un **AUC de 0.6973**, lo que indica una capacidad de discriminación inferior a los modelos de boosting puro, aunque con un tiempo de cómputo sensiblemente reducido.

A continuación, las cinco variables más influyentes según LIME para una instancia de prueba:

| Característica                 | Peso    | Influencia       |
|--------------------------------|--------:|------------------|
| `0.00 < TIP_SS_C <= 1.00`      | –0.100  | Disminuye riesgo |
| `EDAD > 29.00`                 | –0.080  | Disminuye riesgo |
| `1.78 < estrato <= 2.00`       | +0.080  | Aumenta riesgo   |
| `TIP_SS_S <= 0.00`             | +0.080  | Aumenta riesgo   |
| `COD_MUN_R > 436.00`           | –0.020  | Disminuye riesgo |


### **Comparación Final de Modelos en Clasificación Binaria**

A continuación se presenta la tabla comparativa de los cinco modelos binarios ordenados por **F1-score (Macro)**:

| Pos. | Modelo                               | F1 (Macro) | AUC    | CPU Time (s) |
|:----:|:-------------------------------------|:----------:|:------:|:------------:|
| 1    | XGBoost (binario)                    | **0.6792** | 0.8143 | 46.63        |
| 2    | Random Forest (binario)              | 0.6663     | 0.7872 | 193.99       |
| 3    | HistGradientBoosting (binario)       | 0.6647     | 0.8014 | 73.24        |
| 4    | SMOTE + AdaBoost (RandomizedSearchCV)| 0.5858     | 0.6973 | 29.92        |
| 5    | Balanced Random Forest               | 0.5850     | 0.7515 | 45.16        |

**Conclusión:**  
Entre los cinco enfoques evaluados sobre la variable `Victima_Peligro`, XGBoost binario conserva la mejor combinación de capacidad predictiva y eficiencia computacional, liderando en F1-macro (0.6792) y AUC (0.8143) con un tiempo de entrenamiento moderado (46.6 s). Le sigue de cerca Random Forest binario (F1 = 0.6663), que ofrece buena discriminación aunque a costa de un mayor tiempo de cómputo. HistGradientBoosting con early stopping mejora ligeramente la AUC frente a Random Forest, pero su F1-macro es prácticamente equivalente y el tiempo de entrenamiento (73 s) se sitúa en un punto intermedio.  

Los pipelines que combinan SMOTE y AdaBoost, así como el Balanced Random Forest, logran reducir el tiempo de ejecución en algunos casos, pero su F1-score macro queda claramente por debajo de los tres primeros, lo que indica que sacrifican demasiada calidad de detección para ganar velocidad.  

En resumen, XGBoost binario se confirma como la opción más sólida para la detección de víctimas en riesgo en este contexto, equilibrando de forma óptima la sensibilidad (recall), la precisión y la capacidad de respuesta computacional.  